# Stress Prediction v22 — v16 + Physiological Signal Features

Friend got 0.50. After many architecture experiments failed, the most likely missing piece is **better feature engineering on the actual physiological signals** — not more model tweaking.

## What this adds to v16's foundation
- **Frequency-domain HRV** (LF, HF, LF/HF ratio, total power) via Welch's PSD on RR intervals. LF/HF ratio is the classical autonomic stress marker.
- **EDA peak detection** (count of skin conductance responses, mean amplitude, mean width) using scipy `find_peaks` with prominence threshold. SCR rate is a direct sympathetic activation proxy.
- **Resting-baseline-deviation features** — for each subject, identify the bottom 10% of their stream by combined HR+EDA z-score as their resting baseline. Compute window deviations from THAT (not from median, which gets pulled by long stressed periods).
- **Cross-channel correlations** — HR↔EDA correlation in window. Coupled HR+EDA elevation = real stress; HR alone = exercise.
- **Light multi-window** — also compute 1-minute window stats alongside 3-min for fast stress responses.

## What's preserved from v16 (proven 0.377)
- StratifiedKFold 7 seeds × 5 folds = 35 LightGBM models
- `pid_enc` feature
- `class_weight='balanced'` + `sample_weight`
- Calibration `proba × train_prior^1.6`
- Session smoothing strength=0.30

NO heterogeneous ensemble (didn't help in v19/v21). NO calibration changes. ONLY new features.


In [1]:
%pip -q install lightgbm scikit-learn pandas numpy scipy


[notice] A new release of pip is available: 26.0 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from scipy import stats as spstats
from scipy import signal as sps
from scipy.integrate import trapezoid

from sklearn.impute import SimpleImputer
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold

import lightgbm as lgb

warnings.filterwarnings('ignore')
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

DATA_DIR = Path('.')
TRAIN_DATA  = pd.read_csv(DATA_DIR / 'train-sensor.csv')
TRAIN_LABEL = pd.read_csv(DATA_DIR / 'train-label.csv')
TEST_DATA   = pd.read_csv(DATA_DIR / 'test-sensor.csv')
TEST_LABEL  = pd.read_csv(DATA_DIR / 'test-label.csv')

print('Raw shapes')
print('  TRAIN_DATA :', TRAIN_DATA.shape)
print('  TRAIN_LABEL:', TRAIN_LABEL.shape)
print('  TEST_DATA  :', TEST_DATA.shape)
print('  TEST_LABEL :', TEST_LABEL.shape)


Raw shapes
  TRAIN_DATA : (4694400, 8)
  TRAIN_LABEL: (815, 4)
  TEST_DATA  : (5921280, 8)
  TEST_LABEL : (1028, 4)


In [3]:
SENSOR_COLS = ['accel_x', 'accel_y', 'accel_z', 'eda', 'heart_rate', 'temperature']

def clean_sensor(df):
    out = df.copy()
    out['pid'] = out['pid'].astype(str)
    out['timestamp'] = pd.to_numeric(out['timestamp'], errors='coerce').astype(float)
    for c in SENSOR_COLS:
        out[c] = pd.to_numeric(out[c], errors='coerce').astype(float)
    out['accel_x'] = out['accel_x'].clip(-128, 127)
    out['accel_y'] = out['accel_y'].clip(-128, 127)
    out['accel_z'] = out['accel_z'].clip(-128, 127)
    out['eda'] = out['eda'].clip(0, 60)
    out['heart_rate'] = out['heart_rate'].clip(40, 190)
    out['temperature'] = out['temperature'].clip(20, 40)
    return out.sort_values(['pid', 'timestamp']).reset_index(drop=True)

def clean_label(df):
    out = df.copy()
    out['id'] = pd.to_numeric(out['id'], errors='raise').astype(int)
    out['pid'] = out['pid'].astype(str)
    out['timestamp'] = pd.to_numeric(out['timestamp'], errors='coerce').astype(float)
    out['stress'] = pd.to_numeric(out['stress'], errors='coerce')
    return out

TRAIN_DATA  = clean_sensor(TRAIN_DATA)
TEST_DATA   = clean_sensor(TEST_DATA)
TRAIN_LABEL = clean_label(TRAIN_LABEL)
TEST_LABEL  = clean_label(TEST_LABEL)
print('Cleaned.')


Cleaned.


## Compute Resting Baseline Per Subject

For each subject, identify the bottom 10% of their full sensor stream by combined HR + EDA standardized score. Use those samples to compute personal resting baselines (HR, EDA, temperature). Each window's deviation from these is a robust subject-relative stress indicator that doesn't get pulled by long stressed periods (unlike median/MAD).

In [4]:
def compute_resting_baselines(sensor_df, low_pct=10):
    """For each pid, compute resting baseline from low-arousal samples."""
    refs = {}
    for pid, grp in sensor_df.groupby('pid'):
        hr = grp['heart_rate'].values.astype(float)
        eda = grp['eda'].values.astype(float)
        temp = grp['temperature'].values.astype(float)
        valid = np.isfinite(hr) & np.isfinite(eda)
        if valid.sum() < 100:
            refs[pid] = {'hr': float(np.nanmedian(hr)) if valid.any() else 70.0,
                         'eda': float(np.nanmedian(eda)) if valid.any() else 1.0,
                         'temp': float(np.nanmedian(temp)) if valid.any() else 33.0,
                         'hr_std': 5.0, 'eda_std': 0.5}
            continue
        # Standardize, sum, take bottom 10%
        hr_v, eda_v, temp_v = hr[valid], eda[valid], temp[valid]
        hr_z = (hr_v - hr_v.mean()) / (hr_v.std() + 1e-9)
        eda_z = (eda_v - eda_v.mean()) / (eda_v.std() + 1e-9)
        arousal = hr_z + eda_z
        thr = np.percentile(arousal, low_pct)
        rest_mask = arousal < thr
        if rest_mask.sum() < 10:
            rest_mask = np.ones(len(arousal), dtype=bool)
        refs[pid] = {
            'hr':   float(np.median(hr_v[rest_mask])),
            'eda':  float(np.median(eda_v[rest_mask])),
            'temp': float(np.median(temp_v[rest_mask])),
            'hr_std':  float(np.std(hr_v[rest_mask]) + 1e-3),
            'eda_std': float(np.std(eda_v[rest_mask]) + 1e-3),
        }
    return refs

TRAIN_REFS = compute_resting_baselines(TRAIN_DATA)
TEST_REFS  = compute_resting_baselines(TEST_DATA)

print('Train resting baselines:')
for pid, r in TRAIN_REFS.items():
    print(f'  {pid}: rest HR={r["hr"]:.1f}, rest EDA={r["eda"]:.2f}, rest temp={r["temp"]:.2f}')
print('Test resting baselines:')
for pid, r in TEST_REFS.items():
    print(f'  {pid}: rest HR={r["hr"]:.1f}, rest EDA={r["eda"]:.2f}, rest temp={r["temp"]:.2f}')


Train resting baselines:
  43JW: rest HR=72.9, rest EDA=0.42, rest temp=33.99
  C8Q6: rest HR=63.7, rest EDA=0.08, rest temp=30.63
  DT5C: rest HR=71.2, rest EDA=0.21, rest temp=32.57
  F1ZM: rest HR=56.8, rest EDA=0.06, rest temp=29.21
  HDS9: rest HR=71.5, rest EDA=0.52, rest temp=30.49
  P4DZ: rest HR=65.8, rest EDA=0.24, rest temp=28.85
  TPQI: rest HR=74.4, rest EDA=0.52, rest temp=31.91
Test resting baselines:
  01Z2: rest HR=74.1, rest EDA=0.26, rest temp=28.85
  2XO3: rest HR=71.7, rest EDA=0.31, rest temp=34.31
  D1XP: rest HR=72.8, rest EDA=0.62, rest temp=32.21
  NQRB: rest HR=73.6, rest EDA=0.12, rest temp=31.29
  SE4Q: rest HR=74.0, rest EDA=0.57, rest temp=33.27
  SNG7: rest HR=72.7, rest EDA=0.06, rest temp=33.03
  TF0Y: rest HR=73.8, rest EDA=0.90, rest temp=31.37
  Y21H: rest HR=70.7, rest EDA=2.57, rest temp=34.66


## Feature Extraction (v16 features + new physiological features)

In [5]:
WINDOW_MS = 180_000
HALF_MS   = 90_000
THIRD_MS  = 60_000
SHORT_MS  = 60_000  # 1-minute window for fast stress

def hrv_time_domain(bpm_series):
    """v16 HRV time-domain features."""
    f = {}
    bpm = bpm_series.dropna().values.astype(float)
    if len(bpm) < 10:
        for k in ['sdnn','rmssd','pnn25','pnn50','mean_rr','cv_rr']:
            f['hrv_' + k] = np.nan
        return f
    bpm_1hz = bpm[::32] if len(bpm) >= 32 else bpm
    rr = 60000.0 / np.clip(bpm_1hz, 30, 220)
    rr_diff = np.diff(rr)
    f['hrv_sdnn']    = float(np.std(rr))
    f['hrv_rmssd']   = float(np.sqrt(np.mean(rr_diff ** 2))) if len(rr_diff) else 0.0
    f['hrv_pnn25']   = float(np.mean(np.abs(rr_diff) > 25)) * 100 if len(rr_diff) else 0.0
    f['hrv_pnn50']   = float(np.mean(np.abs(rr_diff) > 50)) * 100 if len(rr_diff) else 0.0
    f['hrv_mean_rr'] = float(np.mean(rr))
    f['hrv_cv_rr']   = f['hrv_sdnn'] / f['hrv_mean_rr'] if f['hrv_mean_rr'] > 1e-6 else 0.0
    return f

def hrv_frequency_domain(bpm_series):
    """NEW: Frequency-domain HRV features via Welch's PSD on RR intervals.
    LF (0.04-0.15 Hz): sympathetic + parasympathetic
    HF (0.15-0.40 Hz): parasympathetic / respiratory
    LF/HF ratio: classical sympathovagal balance / stress marker."""
    f = {'hrv_vlf':np.nan, 'hrv_lf':np.nan, 'hrv_hf':np.nan, 'hrv_lf_hf':np.nan, 'hrv_total_power':np.nan}
    bpm = bpm_series.dropna().values.astype(float)
    if len(bpm) < 60:  # need at least 60s of HR data
        return f
    bpm_1hz = bpm[::32] if len(bpm) >= 32 else bpm
    if len(bpm_1hz) < 30:
        return f
    rr = 60000.0 / np.clip(bpm_1hz, 30, 220)  # ms
    rr_centered = rr - rr.mean()
    fs = 1.0  # 1 Hz sampling
    nperseg = min(len(rr_centered), 64)
    if nperseg < 16:
        return f
    try:
        freqs, psd = sps.welch(rr_centered, fs=fs, nperseg=nperseg, 
                               noverlap=nperseg//2, scaling='density')
        def bp(lo, hi):
            mask = (freqs >= lo) & (freqs < hi)
            if mask.sum() < 2: return 0.0
            return float(trapezoid(psd[mask], freqs[mask]))
        f['hrv_vlf'] = bp(0.0033, 0.04)
        f['hrv_lf']  = bp(0.04,   0.15)
        f['hrv_hf']  = bp(0.15,   0.40)
        f['hrv_total_power'] = f['hrv_vlf'] + f['hrv_lf'] + f['hrv_hf']
        f['hrv_lf_hf'] = f['hrv_lf'] / (f['hrv_hf'] + 1e-6)
    except Exception:
        pass
    return f

def eda_peak_features(eda_series, fs=4.0):
    """NEW: Skin conductance response (SCR) detection.
    Returns count, mean amplitude, mean width — direct sympathetic activation markers.
    Note: input may be at 32 Hz; downsample to ~4 Hz for SCR detection."""
    f = {'eda_n_peaks':np.nan, 'eda_peaks_per_min':np.nan, 
         'eda_mean_prominence':np.nan, 'eda_max_prominence':np.nan, 'eda_mean_width':np.nan}
    eda = eda_series.dropna().values.astype(float)
    if len(eda) < 40:
        return f
    # Downsample if oversampled (32 Hz -> 4 Hz)
    if len(eda) >= 100:
        eda_4hz = eda[::8]  # 32/8 = 4 Hz
    else:
        eda_4hz = eda
    if len(eda_4hz) < 16:
        return f
    try:
        # Detrend mildly to focus on transient peaks
        eda_trend = sps.savgol_filter(eda_4hz, window_length=min(15, len(eda_4hz)//2*2+1), polyorder=2) \
                    if len(eda_4hz) > 20 else eda_4hz
        eda_phasic = eda_4hz - eda_trend + np.mean(eda_4hz)
        # Find peaks: prominence >= 0.05 µS, min distance 1 sec
        peaks, props = sps.find_peaks(eda_phasic, prominence=0.02, distance=int(fs*1.0),
                                       width=1)
        f['eda_n_peaks'] = float(len(peaks))
        duration_min = len(eda_4hz) / (fs * 60.0)
        f['eda_peaks_per_min'] = float(len(peaks) / duration_min) if duration_min > 0 else 0.0
        if len(peaks) > 0:
            f['eda_mean_prominence'] = float(np.mean(props['prominences']))
            f['eda_max_prominence']  = float(np.max(props['prominences']))
            f['eda_mean_width']      = float(np.mean(props['widths']))
        else:
            f['eda_mean_prominence'] = 0.0
            f['eda_max_prominence']  = 0.0
            f['eda_mean_width']      = 0.0
    except Exception:
        pass
    return f

def extract_features(label_df, sensor_df, pid_enc_map, refs):
    sensor_by_pid = {pid: grp.sort_values('timestamp').reset_index(drop=True)
                     for pid, grp in sensor_df.groupby('pid')}
    rows = []
    for n, lrow in enumerate(label_df.itertuples(index=False), 1):
        pid = lrow.pid; ts = float(lrow.timestamp); lid = int(lrow.id)
        feat = {'id': lid}
        sg = sensor_by_pid.get(pid)
        if sg is None:
            rows.append(feat); continue
        ta = sg['timestamp'].values
        wa  = sg.loc[(ta >= ts - WINDOW_MS) & (ta <= ts), SENSOR_COLS]
        wf  = sg.loc[(ta >= ts - WINDOW_MS) & (ta < ts - HALF_MS), SENSOR_COLS]
        wl  = sg.loc[(ta >= ts - HALF_MS) & (ta <= ts), SENSOR_COLS]
        wt1 = sg.loc[(ta >= ts - WINDOW_MS) & (ta < ts - 2*THIRD_MS), SENSOR_COLS]
        wt3 = sg.loc[(ta >= ts - THIRD_MS) & (ta <= ts), SENSOR_COLS]
        wshort = sg.loc[(ta >= ts - SHORT_MS) & (ta <= ts), SENSOR_COLS]  # NEW: 1-min
        
        feat['window_count'] = len(wa)
        
        # === v16 standard features ===
        for c in SENSOR_COLS:
            v = wa[c].dropna().values.astype(float)
            vf = wf[c].dropna().values.astype(float)
            vl = wl[c].dropna().values.astype(float)
            vt1 = wt1[c].dropna().values.astype(float)
            vt3 = wt3[c].dropna().values.astype(float)
            if len(v) == 0:
                for s in ['mean','std','min','max','median','skew','kurt','range','q25','q75','iqr','delta','slope','t1_mean','t3_mean','t3t1']:
                    feat[f'{c}_{s}'] = np.nan
                continue
            feat[f'{c}_mean'] = float(np.mean(v))
            feat[f'{c}_std'] = float(np.std(v))
            feat[f'{c}_min'] = float(np.min(v))
            feat[f'{c}_max'] = float(np.max(v))
            feat[f'{c}_median'] = float(np.median(v))
            feat[f'{c}_skew'] = float(spstats.skew(v)) if len(v) > 2 else 0.0
            feat[f'{c}_kurt'] = float(spstats.kurtosis(v)) if len(v) > 2 else 0.0
            feat[f'{c}_range'] = float(np.max(v) - np.min(v))
            feat[f'{c}_q25'] = float(np.percentile(v, 25))
            feat[f'{c}_q75'] = float(np.percentile(v, 75))
            feat[f'{c}_iqr'] = feat[f'{c}_q75'] - feat[f'{c}_q25']
            feat[f'{c}_delta'] = float(np.mean(vl) - np.mean(vf)) if len(vf) and len(vl) else 0.0
            feat[f'{c}_slope'] = float(np.polyfit(np.linspace(0, 1, len(v)), v, 1)[0]) if len(v) > 2 else 0.0
            feat[f'{c}_t1_mean'] = float(np.mean(vt1)) if len(vt1) else float(np.mean(v))
            feat[f'{c}_t3_mean'] = float(np.mean(vt3)) if len(vt3) else float(np.mean(v))
            feat[f'{c}_t3t1'] = feat[f'{c}_t3_mean'] - feat[f'{c}_t1_mean']
        
        # === v16 accel magnitude ===
        ax, ay, az = wa['accel_x'].values, wa['accel_y'].values, wa['accel_z'].values
        if len(ax):
            mag = np.sqrt(ax**2 + ay**2 + az**2)
            feat['accel_mag_mean'] = float(np.mean(mag))
            feat['accel_mag_std']  = float(np.std(mag))
            feat['accel_mag_max']  = float(np.max(mag))
        else:
            feat['accel_mag_mean'] = feat['accel_mag_std'] = feat['accel_mag_max'] = np.nan
        
        # === v16 HRV time domain ===
        feat.update(hrv_time_domain(wa['heart_rate']))
        
        # === NEW: HRV frequency domain ===
        feat.update(hrv_frequency_domain(wa['heart_rate']))
        
        # === NEW: EDA peak features (3-min window) ===
        feat.update(eda_peak_features(wa['eda']))
        
        # === NEW: 1-minute fast-response features (for HR and EDA only — most stress-relevant) ===
        for c in ['heart_rate', 'eda']:
            v_short = wshort[c].dropna().values.astype(float)
            if len(v_short) == 0:
                feat[f'{c}_short_mean'] = feat[f'{c}_short_std'] = feat[f'{c}_short_max'] = np.nan
                feat[f'{c}_short_slope'] = np.nan
                continue
            feat[f'{c}_short_mean'] = float(np.mean(v_short))
            feat[f'{c}_short_std']  = float(np.std(v_short))
            feat[f'{c}_short_max']  = float(np.max(v_short))
            feat[f'{c}_short_slope'] = float(np.polyfit(np.linspace(0,1,len(v_short)), v_short, 1)[0]) if len(v_short) > 2 else 0.0
        
        # === NEW: Resting-baseline-deviation features ===
        ref = refs.get(pid, {})
        if ref:
            hr_mean = feat.get('heart_rate_mean', np.nan)
            eda_mean = feat.get('eda_mean', np.nan)
            temp_mean = feat.get('temperature_mean', np.nan)
            feat['hr_dev_rest']      = (hr_mean - ref['hr'])  if np.isfinite(hr_mean)  else np.nan
            feat['hr_dev_rest_std']  = (hr_mean - ref['hr']) / ref['hr_std']  if np.isfinite(hr_mean) else np.nan
            feat['eda_dev_rest']     = (eda_mean - ref['eda']) if np.isfinite(eda_mean) else np.nan
            feat['eda_dev_rest_std'] = (eda_mean - ref['eda']) / ref['eda_std'] if np.isfinite(eda_mean) else np.nan
            feat['temp_dev_rest']    = (temp_mean - ref['temp']) if np.isfinite(temp_mean) else np.nan
            # Compound stress score: how far above rest in BOTH HR and EDA
            if np.isfinite(feat['hr_dev_rest_std']) and np.isfinite(feat['eda_dev_rest_std']):
                feat['compound_stress'] = feat['hr_dev_rest_std'] + feat['eda_dev_rest_std']
            else:
                feat['compound_stress'] = np.nan
        else:
            for k in ['hr_dev_rest','hr_dev_rest_std','eda_dev_rest','eda_dev_rest_std','temp_dev_rest','compound_stress']:
                feat[k] = np.nan
        
        # === NEW: Cross-channel correlations ===
        try:
            hr = wa['heart_rate'].dropna().values.astype(float)
            eda = wa['eda'].dropna().values.astype(float)
            temp = wa['temperature'].dropna().values.astype(float)
            # Align lengths conservatively
            n_min = min(len(hr), len(eda), len(temp))
            if n_min >= 30:
                hr_a = hr[:n_min]; eda_a = eda[:n_min]; temp_a = temp[:n_min]
                if hr_a.std() > 1e-6 and eda_a.std() > 1e-6:
                    feat['corr_hr_eda'] = float(np.corrcoef(hr_a, eda_a)[0, 1])
                else:
                    feat['corr_hr_eda'] = 0.0
                if hr_a.std() > 1e-6 and temp_a.std() > 1e-6:
                    feat['corr_hr_temp'] = float(np.corrcoef(hr_a, temp_a)[0, 1])
                else:
                    feat['corr_hr_temp'] = 0.0
                if eda_a.std() > 1e-6 and temp_a.std() > 1e-6:
                    feat['corr_eda_temp'] = float(np.corrcoef(eda_a, temp_a)[0, 1])
                else:
                    feat['corr_eda_temp'] = 0.0
            else:
                feat['corr_hr_eda'] = feat['corr_hr_temp'] = feat['corr_eda_temp'] = np.nan
        except Exception:
            feat['corr_hr_eda'] = feat['corr_hr_temp'] = feat['corr_eda_temp'] = np.nan
        
        # pid_enc (kept from v16)
        feat['pid_enc'] = pid_enc_map.get(pid, -1)
        
        rows.append(feat)
        if n % 200 == 0:
            print(f'  {n}/{len(label_df)}')
    return pd.DataFrame(rows).set_index('id')

train_pid_map = {p: i for i, p in enumerate(TRAIN_LABEL['pid'].unique())}
print('Extracting train features...')
train_features = extract_features(TRAIN_LABEL, TRAIN_DATA, train_pid_map, TRAIN_REFS)
print('Extracting test features...')
test_features  = extract_features(TEST_LABEL,  TEST_DATA,  train_pid_map, TEST_REFS)
print(f'\ntrain features: {train_features.shape} (vs v16 ~107)')
print(f'test features:  {test_features.shape}')

# Sanity: show new feature names
new_feats = [c for c in train_features.columns if any(s in c for s in ['hrv_lf','hrv_hf','hrv_vlf','hrv_total','eda_n_peaks','eda_peaks','eda_mean_prom','dev_rest','compound_stress','corr_','_short_'])]
print(f'\nNew features added ({len(new_feats)}):', new_feats)


Extracting train features...
  200/815
  400/815
  600/815
  800/815
Extracting test features...
  200/1028
  400/1028
  600/1028
  800/1028
  1000/1028

train features: (815, 134) (vs v16 ~107)
test features:  (1028, 134)

New features added (25): ['hrv_vlf', 'hrv_lf', 'hrv_hf', 'hrv_lf_hf', 'hrv_total_power', 'eda_n_peaks', 'eda_peaks_per_min', 'eda_mean_prominence', 'heart_rate_short_mean', 'heart_rate_short_std', 'heart_rate_short_max', 'heart_rate_short_slope', 'eda_short_mean', 'eda_short_std', 'eda_short_max', 'eda_short_slope', 'hr_dev_rest', 'hr_dev_rest_std', 'eda_dev_rest', 'eda_dev_rest_std', 'temp_dev_rest', 'compound_stress', 'corr_hr_eda', 'corr_hr_temp', 'corr_eda_temp']


In [6]:
tli = TRAIN_LABEL.set_index('id')
y = tli.loc[train_features.index, 'stress'].astype(int)

# Match columns
common_cols = [c for c in train_features.columns if c in test_features.columns]
train_features = train_features[common_cols]
test_features  = test_features[common_cols]

imputer = SimpleImputer(strategy='median')
X_imp      = pd.DataFrame(imputer.fit_transform(train_features), columns=common_cols, index=train_features.index)
X_test_imp = pd.DataFrame(imputer.transform(test_features),       columns=common_cols, index=test_features.index)

counts = Counter(y); total = len(y)
class_weights = {0: total / (3*counts[0]),
                 1: min(total / (3*counts[1]), 2.5),
                 2: total / (3*counts[2])}
sample_weights = np.array([class_weights[int(yi)] for yi in y])
train_prior = np.array([counts[i]/total for i in range(3)])

print('X_imp:', X_imp.shape)
print('Class weights:', {k:round(v,3) for k,v in class_weights.items()})
print('Train prior:', train_prior.round(3).tolist())


X_imp: (815, 134)
Class weights: {0: 1.677, 1: 2.5, 2: 0.463}
Train prior: [0.199, 0.081, 0.72]


## Sessions + Final Model

Same StratifiedKFold 7×5 LightGBM ensemble as v16. Same calibration.

In [7]:
def make_session_groups(label_df, gap_ms=30 * 60 * 1000):
    labels = label_df.copy().reset_index(drop=True)
    labels['rowpos'] = np.arange(len(labels))
    out = []
    for pid, grp in labels.sort_values(['pid','timestamp']).groupby('pid', sort=False):
        ts = grp['timestamp'].values.astype(float)
        sess = np.cumsum(np.r_[0, np.diff(ts) > gap_ms])
        for sid in np.unique(sess):
            out.append(grp['rowpos'].values[sess == sid])
    return out

def smooth_by_session(proba, sessions, strength=0.30):
    out = proba.copy()
    for idx in sessions:
        mean = proba[idx].mean(axis=0, keepdims=True)
        out[idx] = (1 - strength) * proba[idx] + strength * mean
    return out

train_label_for_rows = TRAIN_LABEL.set_index('id').loc[X_imp.index].reset_index()
TRAIN_SESSIONS = make_session_groups(train_label_for_rows)
TEST_SESSIONS  = make_session_groups(TEST_LABEL)
print('Train sessions:', len(TRAIN_SESSIONS), '| Test sessions:', len(TEST_SESSIONS))

# v16 LGBM params, but slight regularization bump for the larger feature set
LGBM_PARAMS = dict(
    n_estimators=1000, learning_rate=0.02, num_leaves=127, max_depth=-1,
    min_child_samples=10,            # slightly stronger (v16 was 5) for new features
    subsample=0.6, colsample_bytree=0.5,  # slightly more feature subsampling for larger feature set
    reg_alpha=0.3, reg_lambda=0.3,
    class_weight='balanced', objective='multiclass', num_class=3,
    n_jobs=-1, verbose=-1,
)

SEEDS = [42, 7, 123, 17, 99, 256, 314]
N_SPLITS = 5
all_test_proba = []
all_cv_scores = []

for seed in SEEDS:
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    seed_proba = np.zeros((len(X_test_imp), 3))
    fold_scores = []
    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_imp, y), 1):
        model = lgb.LGBMClassifier(**{**LGBM_PARAMS, 'random_state': seed})
        model.fit(
            X_imp.iloc[tr_idx], y.iloc[tr_idx],
            sample_weight=sample_weights[tr_idx],
            eval_set=[(X_imp.iloc[val_idx], y.iloc[val_idx])],
            callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(-1)],
        )
        val_pred = model.predict(X_imp.iloc[val_idx])
        score = balanced_accuracy_score(y.iloc[val_idx], val_pred)
        fold_scores.append(score)
        seed_proba += model.predict_proba(X_test_imp)
    seed_proba /= N_SPLITS
    all_test_proba.append(seed_proba)
    all_cv_scores.append(np.mean(fold_scores))
    print(f'  Seed {seed}: leaky CV={np.mean(fold_scores):.4f}')

raw_test_proba = np.mean(all_test_proba, axis=0)
print(f'\nMean leaky CV: {np.mean(all_cv_scores):.4f} (v16 baseline was 0.804)')
print('Raw test argmax dist:', dict(Counter(raw_test_proba.argmax(1))))

# Show most important features (avg over last seed for diagnostic)
last_model = lgb.LGBMClassifier(**{**LGBM_PARAMS, 'random_state': SEEDS[-1]})
last_model.fit(X_imp, y, sample_weight=sample_weights, callbacks=[lgb.log_evaluation(-1)])
imp = pd.Series(last_model.feature_importances_, index=X_imp.columns).sort_values(ascending=False)
print('\nTop 25 features by importance:')
for name, val in imp.head(25).items():
    flag = ' <-- NEW' if any(s in name for s in ['hrv_lf','hrv_hf','hrv_vlf','hrv_total','eda_n_peaks','eda_peaks','eda_mean_prom','dev_rest','compound_stress','corr_','_short_']) else ''
    print(f'  {name:30s}: {val}{flag}')


Train sessions: 67 | Test sessions: 106
  Seed 42: leaky CV=0.8218
  Seed 7: leaky CV=0.8350
  Seed 123: leaky CV=0.8090
  Seed 17: leaky CV=0.7991
  Seed 99: leaky CV=0.8017
  Seed 256: leaky CV=0.8084
  Seed 314: leaky CV=0.8015

Mean leaky CV: 0.8109 (v16 baseline was 0.804)
Raw test argmax dist: {np.int64(2): 270, np.int64(0): 521, np.int64(1): 237}

Top 25 features by importance:
  temp_dev_rest                 : 1979 <-- NEW
  pid_enc                       : 1040
  temperature_skew              : 755
  heart_rate_min                : 635
  temperature_max               : 600
  accel_z_t3_mean               : 587
  accel_z_mean                  : 558
  eda_dev_rest_std              : 552 <-- NEW
  eda_skew                      : 548
  heart_rate_short_std          : 477 <-- NEW
  temperature_t1_mean           : 466
  eda_min                       : 462
  temperature_mean              : 452
  temperature_min               : 449
  eda_t1_mean                   : 412
  accel_mag_mean

## Calibration + Submission Variants

In [8]:
def make_submission(proba, alpha, smooth_strength, sessions, prior, fname):
    cal = proba * (prior ** alpha)
    cal = cal / cal.sum(axis=1, keepdims=True)
    if smooth_strength > 0:
        cal = smooth_by_session(cal, sessions, strength=smooth_strength)
    preds = np.argmax(cal, axis=1).astype(int)
    pd.DataFrame({'id': TEST_LABEL['id'].values, 'stress': preds}).to_csv(fname, index=False)
    counts = np.bincount(preds, minlength=3)
    fracs  = counts / len(preds)
    dev    = np.abs(fracs - prior).max()
    return preds, counts, fracs, dev

print(f'{"submission":>40s}  {"alpha":>5s} {"smooth":>6s}  {"dist (0/1/2)":>22s}  {"dev":>5s}')
results = {}
for alpha in [1.4, 1.6, 1.8, 2.0]:
    fname = f'submission_alpha_{alpha}.csv'
    preds, counts, fracs, dev = make_submission(raw_test_proba, alpha, 0.30, TEST_SESSIONS, train_prior, fname)
    results[alpha] = (counts, dev)
    print(f'{fname:>40s}  {alpha:>5.2f} {0.30:>6.2f}  {str(counts.tolist()):>22s}  {dev:>5.3f}')

# Default = v16 proven anchor (alpha=1.6, smooth=0.30) but applied to enriched-feature model
preds, counts, fracs, dev = make_submission(raw_test_proba, 1.6, 0.30, TEST_SESSIONS, train_prior, 'submission.csv')
print(f'\n>>> DEFAULT submission.csv (alpha=1.6, smooth=0.30, enriched features)')
print(f'    Distribution : {counts.tolist()}, fractions {fracs.round(3).tolist()}, dev {dev:.3f}')
print(f'    Train prior  : {train_prior.round(3).tolist()}')
print(f'    Target       : ~[0.17, 0.08, 0.75] of 1028 ≈ [170, 80, 770]')


                              submission  alpha smooth            dist (0/1/2)    dev
                submission_alpha_1.4.csv   1.40   0.30          [140, 20, 868]  0.124
                submission_alpha_1.6.csv   1.60   0.30          [100, 10, 918]  0.173
                submission_alpha_1.8.csv   1.80   0.30            [65, 4, 959]  0.213
                submission_alpha_2.0.csv   2.00   0.30            [46, 2, 980]  0.233

>>> DEFAULT submission.csv (alpha=1.6, smooth=0.30, enriched features)
    Distribution : [100, 10, 918], fractions [0.097, 0.01, 0.893], dev 0.173
    Train prior  : [0.199, 0.081, 0.72]
    Target       : ~[0.17, 0.08, 0.75] of 1028 ≈ [170, 80, 770]


In [9]:
print('========== v22 SUMMARY ==========')
print(f'Mean leaky CV BA: {np.mean(all_cv_scores):.4f}  (v16 was ~0.804)')
print()
print('Files saved:')
print('  submission.csv                = alpha=1.6 + smooth=0.30 (DEFAULT, proven calibration)')
print('  submission_alpha_1.4.csv      = lighter calibration')
print('  submission_alpha_1.6.csv      = same as default')
print('  submission_alpha_1.8.csv      = heavier calibration')
print('  submission_alpha_2.0.csv      = heaviest calibration')
print()
print('PICK STRATEGY:')
print('  1. Look at the dist column above.')
print('  2. Submit the file whose dist is closest to train prior [170, 80, 770].')
print('  3. If multiple have similar low dev, prefer alpha=1.6 (proven anchor).')
print('==================================')


========== v22 SUMMARY ==========
Mean leaky CV BA: 0.8109  (v16 was ~0.804)

Files saved:
  submission.csv                = alpha=1.6 + smooth=0.30 (DEFAULT, proven calibration)
  submission_alpha_1.4.csv      = lighter calibration
  submission_alpha_1.6.csv      = same as default
  submission_alpha_1.8.csv      = heavier calibration
  submission_alpha_2.0.csv      = heaviest calibration

PICK STRATEGY:
  1. Look at the dist column above.
  2. Submit the file whose dist is closest to train prior [170, 80, 770].
  3. If multiple have similar low dev, prefer alpha=1.6 (proven anchor).
